# Ordered Logistic Regression Results for Adoption Predictors: Croissant-powered Exploration
This notebook demonstrates how to load, inspect, and analyze the FAIR² dataset of ordered logistic regression outputs for rangeland management practices in Northern Kenya using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a [Croissant](https://mlcommons.org/croissant) schema available at the URL below.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and available record sets from the dataset with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load metadata and dataset object
dataset = mlc.Dataset(croissant_url)

# Show metadata summary
print(f"Dataset Title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
List and inspect available record sets and their fields using the Croissant entity `@id` references.

In [ ]:
# Enumerate record sets and their field @ids
print("Available Record Sets:@ids and Field @ids:")
record_sets_info = []
for record_set in dataset.record_sets:
    print(f"- Record Set @id: {record_set['@id']}")
    record_sets_info.append(record_set['@id'])
    # list fields for the record set
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    if len(fields) == 0:
        print("    (No fields defined)")
    else:
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
            print(f"    - Field @id: {field_id}")

## 3. Data Extraction
Load all available record sets into DataFrames using `@id` references. All identifiers are used dynamically so the notebook adapts automatically if record set or field IDs change.

In [ ]:
# Collect all record set @ids
record_set_ids = [r['@id'] for r in dataset.record_sets]

# Load each record set as DataFrame
dataframes = {}
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    dataframes[rec_id] = pd.DataFrame(records)

# Check for data and preview the first record set
if len(dataframes) == 0:
    print("No record sets found in this dataset.")
else:
    default_rs = record_set_ids[0]
    print(f"First record set: {default_rs}")
    print(f"Columns: {dataframes[default_rs].columns.tolist()}")
    dataframes[default_rs].head()

## 4. Exploratory Data Analysis (EDA)
Perform EDA on a selected numeric field. Steps include filtering by threshold, normalizing, and grouping by a category, using only Croissant `@id` references.

In [ ]:
# EDA for a record set if available
if len(dataframes) > 0:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"DataFrame '{record_set_id}' shape: {df.shape}")

    # Try to select a likely numeric field by scanning columns
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

    if numeric_field is not None:
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df = filtered_df.copy()
        filtered_df[numeric_field + '_normalized'] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Use a categorical/group field if present
        group_field = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
                group_field = col
                break

        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No record sets/dataframes available for EDA.")

## 5. Visualization
Visualize the distribution of values in the numeric field if records exist. All references are as Croissant `@id` fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if len(dataframes) > 0 and 'numeric_field' in locals() and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {record_set_id}")
    plt.xlabel(numeric_field)
    plt.show()
else:
    print("No suitable numeric field for distribution plot.")

## 6. Conclusion
In this notebook, you loaded and explored the FAIR² dataset using Croissant principles. You inspected metadata, extracted record sets using `@id` references, performed exploratory analysis on available fields, and visualized numeric data. 

This workflow ensures reproducibility and schema-driven interoperability for FAIR and responsible data science.